In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
# %%
import datetime
import logging
import os

import pandas as pd
# /venv/lib/python3.12/site-packages/gspread_pandas/spread.py:401: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)` .replace("", np.nan)
pd.set_option('future.no_silent_downcasting', True)

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hpandas as hpandas
import helpers.hprint as hprint
import helpers.hcache as hcache

#hcache.get_global_cache_info()
#hcache.clear_global_cache("all")

import config_root.config as cconfig

# %%
hdbg.init_logger(verbosity=logging.INFO)

_LOG = logging.getLogger(__name__)

_LOG.info("%s", henv.get_system_signature()[0])

hprint.config_notebook()

INFO  > cmd='/venv/lib/python3.12/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-895dfa44-ccb9-4eb9-b5e7-59f0efa12b29.json'
INFO  # Git
  branch_name='CmampTask11020_Compute_yamm_stats'
  hash='6cc14f32e'
  # Last commits:
    * 6cc14f32e GP Saggese Update                                                            ( 5 minutes ago) Sun Jan 19 23:07:31 2025  (HEAD -> CmampTask11020_Compute_yamm_stats, origin/CmampTask11020_Compute_yamm_stats)
    *   ce521e6e2 GP Saggese Merge branch 'master' into CmampTask11020_Compute_yamm_stats      ( 6 minutes ago) Sun Jan 19 23:05:57 2025           
    |\  
    | * 286242538 Shaunak  Dhande checkpoint (#11155)                                               ( 7 minutes ago) Sun Jan 19 23:05:30 2025  (origin/master, origin/HEAD, master)
# Machine info
  system=Linux
  node name=9b21dea71c71
  release=6.10.14-linuxkit
  version=#1 SMP Fri Nov 29 17:22:03 UTC 2024
  machine=aarch64
  processor=aarch64
  cpu count=8
  cp

In [4]:
import gspread
print(gspread.__version__)

import gspread_pandas
print(gspread_pandas.__version__)

#gspread_pandas.conf.get_config()
print(gspread_pandas.conf.get_config()["project_id"])

#!sudo /bin/f bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

import importlib
import ck_marketing.process_automation.hyamm as hyamm
importlib.reload(hyamm)

import ck_marketing.hunterio.hunter_api as cmhuhuap
importlib.reload(cmhuhuap)

import ck_marketing.linkedin.linkedin_utils as cmliliut

#import helpers.hopenai as hopenai

5.12.4
3.3.0
gspread-gp


/app/ck_marketing/process_automation/hyamm.py:19: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# Load data

In [41]:
import helpers.hcache_simple as hscache

@hscache.simple_cache
def _get_cached_sheet_to_df(url: str, sheet_name: str) -> pd.DataFrame:
    _LOG.info(
        "Reading data from url='%s' sheet_name='%s'" % (url, sheet_name)
    )
    spread = gspread_pandas.Spread(url)
    df = spread.sheet_to_df(sheet=sheet_name, index=None)
    return df

In [60]:
url = "https://docs.google.com/spreadsheets/d/13hAmJ58Ois4CcxDoeP-Pwu_JR8kb11wcGHEaDQ5h_Gg"
#df = hyamm.get_cached_sheet_to_df(url, "email_verification")
df = _get_cached_sheet_to_df(url, "email_verification")

hyamm.head(df)

shape= (4878, 9)
columns= ['fullName', 'companyName', 'companyPosition', 'sweetSpot', 'rangePrice', 'firstName', 'lastName', 'hunterio_email', 'hunterio_verification']



,fullName,companyName,companyPosition,sweetSpot,rangePrice,firstName,lastName,hunterio_email,hunterio_verification
0,Ahmed Jawa,Niya Partners,Partner,Sweet spot: $1.5M,Range: $100K - $5.0M,Ahmed,Jawa,a.jawa@niya.vc,valid
1,Aadit Parikh,Sony Innovation Fund,Investor,Sweet spot: $1.5M,Range: $100K - $5.0M,Aadit,Parikh,aadit@sonyinnovationfund.com,accept_all


In [61]:
df["origin"] = "signalnfx_ai_seed"

df2 = hyamm.split_first_last_name(df, "fullName")

for col_name in df2.columns:
    df2[col_name] = df2[col_name].str.strip()

cols_map = {
    "origin": None,
    "first_name": None,
    "last_name": None,
    "companyName": "company_name",
    "companyPosition": "job_title",
    "hunterio_email": "email",
    "hunterio_verification": "email_verification",
}
df2 = hyamm._rename_columns_to_contact_schema(df2, cols_map)
display(df2.head())

,fullName,first_name,last_name,company_name,job_title,sweetSpot,rangePrice,firstName,lastName,email,email_verification,origin
0,Ahmed Jawa,Ahmed,Jawa,Niya Partners,Partner,Sweet spot: $1.5M,Range: $100K - $5.0M,Ahmed,Jawa,a.jawa@niya.vc,valid,signalnfx_ai_seed
1,Aadit Parikh,Aadit,Parikh,Sony Innovation Fund,Investor,Sweet spot: $1.5M,Range: $100K - $5.0M,Aadit,Parikh,aadit@sonyinnovationfund.com,accept_all,signalnfx_ai_seed
2,Ajay Agarwal,Ajay,Agarwal,Bain Capital Ventures,Partner,Sweet spot: $1.5M,Range: $100K - $100.0M,Ajay,Agarwal,aagarwal@baincapitalventures.com,valid,signalnfx_ai_seed
3,Aaron Applbaum,Aaron,Applbaum,MizMaa Ventures,Managing Partner,Sweet spot: $3.0M,Range: $1M - $6.0M,Aaron,Applbaum,aaron.applbaum@mizmaa.com,accept_all,signalnfx_ai_seed
4,Aaron Tay,Aaron,Tay,Auspac Investment Management,Investor,Sweet spot: $2.5M,Range: $1M - $5.0M,Aaron,Tay,aaron.tay@auspacim.com,valid,signalnfx_ai_seed
